In [ ]:
import math

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sympy as sym

from rotation.symbolic import ANGLE, BARE, PROPAGATION, create_3d_rotation
from rotation.symbolic_axial import create_kinetic, create_axial_mix_equation
from rotation.solver import initialize_system, create_func, solve_mixing, calculate_uvw

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    PROPAGATION["pi0"]: 0.1349768,
    PROPAGATION["eta"]: 0.547862,
}

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [PROPAGATION["pi0"], BARE["pi0"]]:
        value *= sym.Symbol(r" m_{\pi^0\text{, PDG}}") / PDG[PROPAGATION["pi0"]]
    elif key in [PROPAGATION["eta"], BARE["eta"]]:
        value *= sym.Symbol(r" m_{\eta\text{, PDG}}") / PDG[PROPAGATION["eta"]]
    elif key in [PROPAGATION["phi"], PROPAGATION["eta0"], BARE["phi"], BARE["pi0-phi"], BARE["eta-phi"], BARE["eta0"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ANGLE.values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

## Prepare solver

In [ ]:
rotate = create_3d_rotation()[0]
kin = create_kinetic()
equation = create_axial_mix_equation(rotate, kin)[0]
argument_order, system_of_equations = initialize_system(equation.lhs - equation.rhs)

## Solver inputs

In [ ]:
kin_u, kin_v, kin_w = calculate_uvw(
    m_axial=5,
    m_pi0_axial=PDG[PROPAGATION["pi0"]],
    m_eta_axial=PDG[PROPAGATION["eta"]],
    m_eta0_axial=PDG[PROPAGATION["eta"]]*0.9,
)
# Fixed known parameters
known = {
    PROPAGATION["pi0"]: PDG[PROPAGATION["pi0"]],
    PROPAGATION["eta"]: PDG[PROPAGATION["eta"]],
    PROPAGATION["eta0"]: PDG[PROPAGATION["eta"]]*0.9,
    ANGLE["u"]: kin_u,
    ANGLE["v"]: kin_v,
    ANGLE["w"]: kin_w,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    BARE["pi0"]: PDG[PROPAGATION["pi0"]],
    BARE["eta"]: PDG[PROPAGATION["eta"]],
    BARE["eta0"]: PDG[PROPAGATION["eta"]]*0.9,
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,
}

# Printout the system with our known values, and unknowns blank
print("Arguments input:")
for k in argument_order:
    if k in known:
        sym.Eq(k, known[k], evaluate=False)
    else:
        sym.Eq(k, sym.Symbol("?"), evaluate=False)
equation.subs(known)

## Run solver

In [ ]:
# Create function
func = create_func(
    system_of_equations,
    argument_order,
    known,
)
try:
    # Solve system
    solved = solve_mixing(
        func,
        argument_order,
        unknown,
    )
    print("Solver succeeded!")
except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")

## Solution display

In [ ]:
# Printout the system with our solved unknowns, and knowns blank
print("Arguments solved:")
for k, v in solved.items():
    v = format_output(k, v)
    sym.Eq(k, v, evaluate=False)

print("The solved equation:")
equation.subs(solved)

print("Compare LHS and RHS evaluations:")
sym.Eq(
    (1*equation.lhs.subs(known|solved)).evalf(),
    (1*equation.rhs.subs(known|solved)).evalf(),
    evaluate=False
)